# Base-paper replication (Zhao, Fok & Thing 2024)
Run the cells top to bottom. `paper_replica.py` must be in the same folder as this notebook.
The full run executes in the **background**, so it keeps going if the browser or kernel disconnects. Re-run the monitor cell any time.

## 1. Settings

In [ ]:
import os
DATA_DIR = "/path/to/MachineLearningCVE"   # <-- folder with the 8 CSV files
OUT      = "paper_replica_out"
GPU      = "2"                              # free GPU id on the DGX, or "" for CPU
TREES    = 100                              # keep 100 for fidelity; 50 is faster
STEPS    = 3000                             # GAN iterations per segment
NVFIX   = os.path.expanduser("~/nvfix/nvidia/nvjitlink/lib/libnvJitLink.so.12")  # torch library fix

## 2. Environment check

In [ ]:
import sys, subprocess, glob, time
import pandas as pd
ENV = dict(os.environ, CUDA_VISIBLE_DEVICES=GPU, LD_PRELOAD=NVFIX)   # applied to every script run below
assert os.path.exists("paper_replica.py"), "paper_replica.py not found next to this notebook"
if not os.path.exists(NVFIX): ENV.pop("LD_PRELOAD", None)   # optional torch library fix; not needed on most systems
r = subprocess.run([sys.executable, "-c", "import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())"],
                   capture_output=True, text=True, env=ENV)
print(r.stdout or r.stderr[-800:])
files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
print(len(files), "CSV files found"); [print("  ", os.path.basename(f)) for f in files]

## 3. (Optional) 1-minute smoke test on fake data

In [ ]:
r = subprocess.run([sys.executable, "paper_replica.py", "--demo"], capture_output=True, text=True, env=ENV)
print(r.stdout[-1500:]); print(r.stderr[-1500:])

## 4. Launch the full run in the background

In [ ]:
cmd = [sys.executable, "paper_replica.py", "--data_dir", DATA_DIR, "--out", OUT,
       "--trees", str(TREES), "--steps", str(STEPS)]
log = open("run.log", "a")
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=ENV, start_new_session=True)
print("started, pid", proc.pid)

## 5. Monitor progress (re-run as often as you like)

In [ ]:
print("".join(open("run.log").readlines()[-30:]))
done = os.path.exists(f"{OUT}/tables/table8_wgan99_all_classes.csv")
print("\nFINISHED" if done else "\nstill running (or stopped; re-run step 4 to resume from cache)")

## 6. Tables: ours vs paper

In [ ]:
from IPython.display import display, Markdown
T = f"{OUT}/tables"
def show(title, name):
    p = f"{T}/{name}.csv"
    display(Markdown(f"### {title}"))
    display(pd.read_csv(p)) if os.path.exists(p) else print("not ready yet:", p)
show("Table 1: class counts (diff must be 0)", "table1_counts")
show("Table 3: regrouped classes", "table3_regrouped")
show("Table 4: baseline NIDS (RF)", "table4_baseline")
show("Table 5: cosine similarity (rows marked 'paper' are published values)", "table5_cosine")
show("Section 3.6.3: ML validation of generated samples", "ml_validation")

In [ ]:
show("Table 6: IDS on generated Botnet samples", "table6_generated_test")
show("Table 7: trained NIDS tested on ORIGINAL Botnet samples", "table7_original_botnet")
show("Table 8: WGAN x99, all classes", "table8_wgan99_all_classes")

## 7. Figures 2-4: cumulative sums

In [ ]:
from IPython.display import Image, display
for f in ["fig2_vanilla_gan", "fig3_wgan", "fig4_ctgan"]:
    p = f"{OUT}/figures/{f}.png"
    if os.path.exists(p): display(Markdown(f"### {f}")); display(Image(p, width=800))